## 🌾 Machine Learning: Crop Recommendation System

### 📚 Deskripsi Dataset

Pada studi kasus ini, kita akan membuat **sistem rekomendasi tanaman** berbasis **Machine Learning** yang bertujuan membantu petani dalam menentukan jenis tanaman terbaik berdasarkan kondisi lingkungan dan tanah.

Dataset yang digunakan berisi **data agrikultur** dengan berbagai parameter penting yang mempengaruhi pertumbuhan tanaman. Data ini mencakup **unsur hara tanah (N, P, K)**, **suhu**, **kelembaban udara**, **tingkat keasaman tanah (pH)**, dan **curah hujan**. Berdasarkan parameter-parameter ini, sistem akan memberikan **rekomendasi jenis tanaman** yang paling sesuai.

Berikut adalah deskripsi dari setiap kolom dalam dataset:

| **Kolom**        | **Tipe Data**          | **Deskripsi**                                                                                     |
|------------------|------------------------|---------------------------------------------------------------------------------------------------|
| **N**            | `int`                  | Kandungan Nitrogen dalam tanah, diukur dalam satuan mg/kg.                                       |
| **P**            | `int`                  | Kandungan Phosphorus dalam tanah, diukur dalam satuan mg/kg.                                     |
| **K**            | `int`                  | Kandungan Kalium dalam tanah, diukur dalam satuan mg/kg.                                      |
| **temperature**  | `float`                | Suhu lingkungan tempat tanaman tumbuh, diukur dalam derajat Celcius (°C).                       |
| **humidity**     | `float`                | Kelembaban udara di lingkungan tumbuh, diukur dalam persen (%).                                  |
| **ph**           | `float`                | Tingkat keasaman tanah (pH), menunjukkan kondisi asam atau basa pada tanah.                      |
| **rainfall**     | `float`                | Curah hujan tahunan di wilayah tanam, diukur dalam milimeter (mm).                               |
| **label**        | `category` / `string`  | Jenis tanaman yang direkomendasikan untuk ditanam berdasarkan parameter yang ada.                |

---

### 🚀 Workflow Proyek
1. **Import Library & Read Data**  
2. **Exploratory Data Analysis**  
3. **Feature Selection**  
4. **Model Selection & Training**  
5. **Inference**  

---

### 🔧 Langkah-Langkah Penerapan Machine Learning


---


## Import Libraries and Read Data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("Crop_recommendation.csv")
df.head()

📌 __Penjelasan__:

Pada kode diatas, kita melakukan import library - library yang dibutuhkan pada project ini, kemudian membaca dataset.

In [ ]:
df = df.rename(columns={"N":"nitrogen", "P":"phosphorus", "K":"kalium"})
df.head()

📌 __Penjelasan__:

Kita dapat mengubah nama kolom menggunakan method rename pada dataframe 

## Exploratory Data Analysis

In [ ]:
df.info()

📌 __Penjelasan__:

Pada kode tersebut, kita ingin melihat gambaran umum dari data yang kita miliki. Informasi yang ditampilkan meliputi:

1. Jumlah baris dan kolom pada dataset.
2. Nama setiap kolom beserta jumlah data yang tidak kosong.
3. Tipe data dari masing-masing kolom, seperti angka (```int64```, ```float64```) atau kategori (```category```).
4. Penggunaan memori dari data tersebut.

Dengan ```df.info()```, kita bisa mengecek apakah ada data kosong (```null```) dan memastikan tipe data sudah sesuai, sebelum masuk ke tahap analisis lebih lanjut.

In [ ]:
## Check for Null Values
print("Null values in the dataset:")
print(df.isnull().sum())

print("\nTotal null values:", df.isnull().sum().sum())

## Replace null values with mean (if any exist)
# df = df.fillna(df.mean())


📌 __Penjelasan__:

Pada kode di atas, kita melakukan pengecekan apakah ada data yang kosong (```null```) dalam dataset.

```df.isnull().sum()``` untuk Menghitung jumlah nilai kosong di tiap kolom.

```df.isnull().sum().sum()``` untuk Menghitung total nilai kosong di seluruh dataset.

Jika ditemukan nilai kosong, kita bisa mengisinya dengan nilai rata-rata menggunakan ```df.fillna(df.mean())``` agar data menjadi lengkap dan bisa digunakan untuk analisis atau pemodelan machine learning.

In [ ]:
## Check for Duplicates
print("Number of duplicate rows:", df.duplicated().sum())

## Remove Duplicates 
#df.drop_duplicates()

📌 __Penjelasan__:

Pada kode tersebut, kita melakukan pengecekan data duplikat dalam dataset.

```df.duplicated()``` untuk mengecek setiap baris apakah ada yang sama persis dengan baris lain.

```.sum()``` untuk menghitung jumlah baris duplikat yang ditemukan.

Duplikat bisa menyebabkan analisis menjadi tidak akurat, jadi penting untuk mendeteksinya sebelum proses analisis atau pemodelan. Jika ada duplikat, biasanya kita hapus dengan ```df.drop_duplicates()```.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



## Check for Outliers using Box Plots
plt.figure(figsize=(20,10))

# Create subplots for each numerical column
for i, column in enumerate(['nitrogen', 'phosphorus', 'kalium', 'temperature', 'humidity', 'ph', 'rainfall'], 1):
    plt.subplot(2, 4, i)
    sns.boxplot(y=df[column], color='skyblue')
    plt.title(f'Boxplot of {column}')

plt.tight_layout()
plt.show()


📌 __Penjelasan__:

Pada kode di atas, kita mengimpor library visualisasi yaitu ```seaborn``` dan ```matplotlib.pyplot``` untuk membuat grafik.

Kemudian, kita melakukan deteksi outlier (nilai pencilan) menggunakan boxplot untuk setiap kolom numerik pada dataset (```N```, ```P```, ```K```, ```temperature```, ```humidity```, ```ph```, ```rainfall```).

```plt.figure(figsize=(20,10))``` untuk mengatur ukuran figure supaya lebih lebar.

```for i, column in enumerate(...):``` untuk membuat loop pada pembuatan boxplot di setiap kolom numerik.

```sns.boxplot()``` untuk membuat boxplot dari kolom yang sedang diproses.

```plt.title()``` untuk memberikan judul pada masing-masing plot.

```plt.tight_layout()``` untuk mengatur tata letak agar plot tidak saling tumpang tindih.

```plt.show()``` untuk menampilkan seluruh plot dalam satu gambar.

Tujuan boxplot adalah untuk melihat distribusi data dan mendeteksi outlier secara visual.

In [ ]:
## Outlier Detection using IQR
def detect_outliers(df, column):
    # 1. Hitung Q1 (persentil 25) dan Q3 (persentil 75)
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    # 2. Hitung IQR (Interquartile Range)
    IQR = Q3 - Q1
    
    # 3. Tentukan batas bawah dan batas atas (aturan 1.5 * IQR)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)][column]
    return len(outliers)

numerical_columns = ['nitrogen', 'phosphorus', 'kalium', 'temperature', 'humidity', 'ph', 'rainfall']
for column in numerical_columns:
    n_outliers = detect_outliers(df, column)
    print(f"Number of outliers in {column}: {n_outliers}")

<img src="https://github.com/epsindo-ai/bootcamp_PNP2/blob/main/training-d3/sesi_1/iqr.png?raw=true" alt="Boxplot Outlier Detection">

📌 __Penjelasan__:

Pada kode di atas, kita melakukan deteksi outlier menggunakan metode __IQR (Interquartile Range)__.

Fungsi ```detect_outliers()``` digunakan untuk menghitung berapa banyak outlier di setiap kolom numerik.

```Q1``` dan ```Q3``` Mengambil kuartil pertama (25%) dan kuartil ketiga (75%).

```IQR``` Selisih antara Q3 dan Q1 (IQR = Q3 - Q1).

```lower_bound``` dan ```upper_bound``` Batas bawah dan batas atas. Nilai di luar batas ini dianggap outlier.

Baris yang nilainya lebih kecil dari batas bawah atau lebih besar dari batas atas dikategorikan sebagai outlier.
Loop for akan mengecek setiap kolom numerik dalam list ```numerical_columns```.

```print()``` akan menampilkan jumlah outlier yang ditemukan di setiap kolom.

Tujuan kode ini adalah untuk membantu kita mengetahui apakah ada nilai ekstrem dalam data yang bisa mempengaruhi analisis atau model machine learning.

✅ Cara penanganan outlier tergantung kasus:

1. Hapus, jika outlier adalah data error.
2. Imputasi, jika outlier relevan tapi ekstrem.

In [ ]:
def handle_outliers(df, kolom_list, metode="median"):
    """
    Menangani outlier dalam beberapa kolom dengan opsi mengganti dengan median atau menghapusnya.

    Return:
    DataFrame yang sudah dibersihkan.
    """

    for kolom in kolom_list:
        # 1. Hitung Q1 (persentil 25) dan Q3 (persentil 75)
        Q1 = df[kolom].quantile(0.25)
        Q3 = df[kolom].quantile(0.75)

        # 2. Hitung IQR (Interquartile Range)
        IQR = Q3 - Q1

        # 3. Tentukan batas bawah dan batas atas (aturan 1.5 * IQR)
        batas_bawah = Q1 - 1.5 * IQR
        batas_atas = Q3 + 1.5 * IQR

        if metode == "median":
            # Ganti outlier dengan median
            nilai_median = df[kolom].median()
            # Bulatkan ke integer untuk kolom yang bertipe int
            if df[kolom].dtype == 'int64':
                nilai_median = int(round(nilai_median))
            df.loc[(df[kolom] < batas_bawah) | (df[kolom] > batas_atas), kolom] = nilai_median
            
        elif metode == "mean":
            # Ganti outlier dengan mean
            nilai_mean = df[kolom].mean()
            # Bulatkan ke integer untuk kolom yang bertipe int
            if df[kolom].dtype == 'int64':
                nilai_mean = int(round(nilai_mean))
            df.loc[(df[kolom] < batas_bawah) | (df[kolom] > batas_atas), kolom] = nilai_mean

        elif metode == "hapus":
            # Hapus baris yang mengandung outlier
            df = df[~((df[kolom] < batas_bawah) | (df[kolom] > batas_atas))]

    return df

# Contoh penggunaan
kolom_outlier = ['humidity', 'ph', 'rainfall', 'temperature', 'phosphorus', 'kalium']
df_mean = handle_outliers(df, kolom_outlier, metode="mean") # Ganti outlier dengan mean
# df_median = handle_outliers(df, kolom_outlier, metode="median")  # Ganti outlier dengan median
# df_clean = handle_outliers(df, kolom_outlier, metode="hapus")  # Hapus outlier


In [ ]:
df.describe()

📌 __Penjelasan__:

Pada kode ```df.describe()```, kita melakukan ringkasan statistik deskriptif terhadap kolom-kolom numerik dalam dataset.

In [ ]:
df.columns

📌 __Penjelasan__:

Pada kode ```df.columns```, kita memeriksa nama kolom apa saja yang ada pada dataset.

In [ ]:
df.shape

📌 __Penjelasan__:

Pada kode ```df.shape```, kita memeriksa jumlah dari dataset (banyak baris, banyak kolom).

In [ ]:
df['label'].unique()

📌 __Penjelasan__:

Kode ```df['label'].unique()``` digunakan untuk menampilkan semua nilai unik (distinct values) yang terdapat pada kolom ```label``` di dataset.

In [ ]:
df['label'].nunique()

📌 __Penjelasan__:

Kode ```df['label'].nunique()``` digunakan untuk menghitung jumlah nilai unik atau kategori berbeda yang ada di kolom label.

In [ ]:
df['label'].value_counts()

📌 __Penjelasan__:

Kode ```df['label'].value_counts()``` digunakan untuk menghitung jumlah kemunculan (frekuensi) dari setiap kategori unik yang ada di kolom label.

In [ ]:
plt.figure(figsize=(12,5))

# Temperature distribution
plt.subplot(1, 2, 1)
sns.histplot(df['temperature'], color="red", bins=15, kde=True, alpha=0.5)

# pH distribution
plt.subplot(1, 2, 2)
sns.histplot(df['ph'], color="green", bins=15, kde=True, alpha=0.5)

📌 __Penjelasan__:

Kode di atas digunakan untuk membuat visualisasi distribusi data pada dua variabel berbeda, yaitu ```temperature``` dan ```ph```, dengan histogram dan kurva KDE (Kernel Density Estimation) secara berdampingan.

In [ ]:
sns.jointplot(x="rainfall",y="humidity",data=df[(df['temperature']<40) & 
                                                  (df['rainfall']>40)],height=10,hue="label")

📌 __Penjelasan__:

Kode di atas digunakan untuk membuat visualisasi hubungan (relasi) antara dua variabel numerik ```rainfall``` dan ```humidity``` menggunakan ```sns.jointplot()```, yang secara otomatis menampilkan scatterplot di tengah, dan distribusi (histogram) di sumbu X dan Y.

Selain itu, grafik ini juga dikelompokkan berdasarkan kategori tanaman (label) agar lebih informatif.

In [ ]:
sns.pairplot(df,hue = 'label')

📌 __Penjelasan__:

Kode ```sns.pairplot(df, hue='label')``` digunakan untuk membuat visualisasi multivariat dari seluruh kombinasi fitur numerik di dataset, dengan membedakan kelompok berdasarkan nilai dari kolom ```label```.

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(30,15))
sns.boxplot(x='label',y='ph',data=df)

📌 __Penjelasan__:

Kode berikut digunakan untuk membuat visualisasi boxplot dari kolom ```ph``` untuk masing-masing label tanaman, dengan tema tampilan grafis yang lebih estetik menggunakan Seaborn.

Fungsi dan Tujuan Visualisasi Ini adalah Untuk melihat penyebaran nilai ```pH``` pada masing-masing jenis tanaman (```label```), Jenis tanaman mana yang membutuhkan pH lebih tinggi/lebih rendah.

In [ ]:
df_corr = df.drop(columns=["label"])

fig, ax = plt.subplots(1, 1, figsize=(15, 9))
sns.heatmap(df_corr.corr(), annot=True,cmap='viridis')
ax.set(xlabel='features')
ax.set(ylabel='features')

plt.title('Correlation between different features', fontsize = 15, c='black')
plt.show()

📌 __Penjelasan__:

Kode berikut digunakan untuk membuat heatmap korelasi antar fitur numerik dalam dataset, dengan label (```target```) dihapus terlebih dahulu karena bukan variabel numerik.

Fungsi dan Tujuan Visualisasi Ini:
1. Untuk memahami hubungan linear antar fitur numerik.
2. Nilai korelasi berkisar antara -1 hingga 1:
  * 1 → Korelasi positif sempurna (jika satu naik, yang lain ikut naik).
  * -1 → Korelasi negatif sempurna (jika satu naik, yang lain turun).
  * 0 → Tidak ada korelasi.
  

Membantu analisis Feature Selection yaitu memilih fitur penting untuk model machine learning.


In [ ]:
df_summary = pd.pivot_table(df, index=['label'], aggfunc='mean')
df_summary.head()

📌 __Penjelasan__:

Pada kode di atas, kita membuat sebuah pivot table dari dataframe df untuk menganalisis nilai rata-rata tiap label (jenis tanaman).

In [ ]:

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_summary.index,
    y=df_summary['nitrogen'],
    name='Nitrogen',
    marker_color='mediumvioletred'
))
fig.add_trace(go.Bar(
    x=df_summary.index,
    y=df_summary['phosphorus'],
    name='Phosphorous',
    marker_color='springgreen'
))
fig.add_trace(go.Bar(
    x=df_summary.index,
    y=df_summary['kalium'],
    name='Kalium',
    marker_color='dodgerblue'
))

fig.update_layout(title="N-P-K values comparision between crops",
                  plot_bgcolor='white',
                  barmode='group',
                  xaxis_tickangle=-45)

fig.show()

📌 __Penjelasan__:

Kode di atas digunakan untuk membuat visualisasi perbandingan kandungan N-P-K (Nitrogen, Phosphorus, Kalium) pada berbagai tanaman, menggunakan Plotly.

Untuk membandingkan kebutuhan Nutrisi Tanaman pada berbagai jenis tanaman dan membantu memahami tanaman mana yang membutuhkan Nitrogen, Phosphorous, atau Potash lebih tinggi.

## Feature Selection

In [ ]:
# Import LabelEncoder
from sklearn.preprocessing import LabelEncoder

# Create features (numerical columns only)
features = df.drop(columns=['label'])

# Create target variable (original string labels)
target = df['label']

# Create label encoder for target variable
label_encoder = LabelEncoder()
target_encoded = label_encoder.fit_transform(target)

# Create mapping dictionaries for reference
label_to_number = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# Display the label mapping
print("Label Encoding Mapping:")
for label, number in sorted(label_to_number.items()):
    print(f'"{label}": {number}')

In [ ]:
acc = []
model = []

In [ ]:
from sklearn.model_selection import train_test_split

# Split using encoded labels for model training
x_train, x_test, y_train, y_test = train_test_split(features, target_encoded, test_size=0.2, random_state=42)

📌 __Penjelasan__:

Kode ini mempersiapkan data untuk machine learning:

1. **Features**: Memisahkan kolom numerik (```features```) dari kolom target (```label```).
2. **Target Encoding**: Menggunakan ```LabelEncoder``` untuk mengkonversi label string menjadi angka:
   - String labels (apple, banana, rice, dll.) → Angka (0, 1, 2, dll.)
   - Menyimpan mapping dalam dictionary untuk referensi
3. **Train-Test Split**: Membagi data menjadi data training (80%) dan testing (20%) dengan ```random_state=42``` untuk reproducibility.
4. **Correlation Analysis**: Menambahkan target yang sudah di-encode ke dalam analisis korelasi untuk melihat hubungan fitur dengan target.

**Keuntungan Label Encoding**:
- Machine learning algorithms lebih efisien dengan angka
- Dapat dianalisis korelasinya dengan fitur numerik lainnya
- Tetap bisa dikonversi kembali ke label string untuk interpretasi hasil

Variabel ```acc``` dan ```model``` digunakan untuk menyimpan akurasi dan nama model untuk perbandingan nanti.

## 🚀Model Selection & Training

### Classification Algorithms

#### K-Nearest Neighbor

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.metrics import classification_report

knn = KNeighborsClassifier()

knn.fit(x_train,y_train)

predicted_values = knn.predict(x_test)

x = metrics.accuracy_score(y_test, predicted_values)
acc.append(x)
model.append('K Nearest Neighbours')
print("KNN Accuracy is: ", x)

print(classification_report(y_test,predicted_values))

📌 __Penjelasan__:

Kode ini membuat, melatih, dan mengevaluasi model K-Nearest Neighbors untuk klasifikasi tanaman berdasarkan dataset yang sudah disiapkan sebelumnya. Setelah model diprediksi, akurasinya diukur dan performa detail tiap kelas dievaluasi lewat classification_report.

In [ ]:
from sklearn.model_selection import cross_val_score

score = cross_val_score(knn, x_train, y_train,cv=5)
print('Cross validation score: ',score)

📌 __Penjelasan__:

```cross_val_score()``` akan:

1. Membagi dataset (features dan target) menjadi 5 bagian (folds) karena cv=5.
2. Melatih dan menguji model knn sebanyak 5 kali, setiap kali menggunakan fold yang berbeda sebagai data uji, dan sisanya sebagai data latih.
3. Mengembalikan 5 skor akurasi (satu untuk tiap fold).

In [ ]:
#Print Train Accuracy
knn_train_accuracy = knn.score(x_train,y_train)
print("knn_train_accuracy = ",knn.score(x_train,y_train))
#Print Test Accuracy
knn_test_accuracy = knn.score(x_test,y_test)
print("knn_test_accuracy = ",knn.score(x_test,y_test))

📌 __Penjelasan__:

```knn.score(x_train, y_train)``` menghitung akurasi model KNN pada data latih (train set).

```knn.score(x_test, y_test)``` menghitung akurasi model KNN pada data uji (test set).

* Train Accuracy terlalu tinggi + Test Accuracy rendah → indikasi overfitting (model hafal data train tapi jelek di data baru).
* Train & Test Accuracy sama-sama rendah → kemungkinan underfitting (model belum cukup belajar pola data).
* Train & Test Accuracy seimbang dan cukup tinggi → model punya generalization yang baik.

In [ ]:
y_pred = knn.predict(x_test)
y_true = y_test

from sklearn.metrics import confusion_matrix

cm_knn = confusion_matrix(y_true, y_pred)

# Use existing label mapping dictionary
labels = list(label_to_number.keys())

f, ax = plt.subplots(figsize=(15,10))
sns.heatmap(cm_knn, annot=True, linewidth=0.5, cmap='viridis', ax=ax,
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Predicted vs Actual (KNN Model)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

📌 __Penjelasan__:

Membuat Confusion Matrix (```cm_knn```), yaitu tabel yang menunjukkan jumlah prediksi benar dan salah model untuk setiap kelas.

#### Hyperparameter Tuning

In [ ]:
mean_acc = np.zeros(20)
for i in range(1,21):
    #Train Model and Predict  
    knn = KNeighborsClassifier(n_neighbors = i).fit(x_train,y_train)
    yhat= knn.predict(x_test)
    mean_acc[i-1] = metrics.accuracy_score(y_test, yhat)

mean_acc

📌 __Penjelasan__:

Untuk menguji performa model K-Nearest Neighbors (KNN) dengan berbagai jumlah tetangga terdekat (```n_neighbors```) dan mengetahui berapa nilai K terbaik agar akurasi model paling tinggi.

Nilai akurasi terbaik ada di ```n_neighbors``` = 3 dan ```n_neighbors``` = 5, dengan akurasi 97.04% (0.97045455).

In [ ]:
loc = np.arange(1,21,step=1.0)
plt.figure(figsize = (10, 6))
plt.plot(range(1,21), mean_acc)
plt.xticks(loc)
plt.xlabel('Number of Neighbors ')
plt.ylabel('Accuracy')
plt.show()

📌 __Penjelasan__:

Plot akurasi model KNN untuk tiap jumlah neighbour ```K```.

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_params = { 'n_neighbors' : [12,13,14,15,16,17,18],
               'weights' : ['uniform','distance'],
               'metric' : ['minkowski','euclidean','manhattan']}

gs = GridSearchCV(KNeighborsClassifier(), grid_params, verbose = 1, cv=3, n_jobs = -1)

g_res = gs.fit(x_train, y_train)

print("Best Score: ",g_res.best_score_)
print("Best Parameter: ",g_res.best_params_)

📌 __Penjelasan__:

Kode ini bertujuan mencari kombinasi hyperparameter terbaik untuk model K-Nearest Neighbors (KNN) menggunakan ```GridSearchCV```.
Hyperparameter tuning ini membantu mendapatkan model dengan akurasi optimal.

```GridSearchCV``` mencari kombinasi hyperparameter terbaik dengan cara mengevaluasi performa model berdasarkan metrik evaluasi di setiap kombinasi yang dicoba.

In [ ]:
knn_1 = KNeighborsClassifier(n_neighbors = 12, weights = 'distance',algorithm = 'brute',metric = 'manhattan')
knn_1.fit(x_train, y_train)

📌 __Penjelasan__:

Melakukan modeling dengan menggunakan parameter yang optimal.

In [ ]:
# Training & Testing accuracy after applying hyper parameter
knn_train_accuracy = knn_1.score(x_train,y_train)
print("knn_train_accuracy = ",knn_1.score(x_train,y_train))
#Print Test Accuracy
knn_test_accuracy = knn_1.score(x_test,y_test)
print("knn_test_accuracy = ",knn_1.score(x_test,y_test))

📌 __Penjelasan__:

Melihat hasil evaluasi akurasi performa model yang telah dibangun.

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT = DecisionTreeClassifier(criterion="entropy",random_state=2,max_depth=10)

DT.fit(x_train,y_train)

predicted_values = DT.predict(x_test)
x = metrics.accuracy_score(y_test, predicted_values)
acc.append(x)
model.append('Decision Tree')
print("Decision Tree's Accuracy is: ", x*100)

print(classification_report(y_test,predicted_values))

In [ ]:
score = cross_val_score(DT, features, target,cv=5)
print('Cross validation score: ',score)

#Print Train Accuracy
dt_train_accuracy = DT.score(x_train,y_train)
print("Training accuracy = ",DT.score(x_train,y_train))
#Print Test Accuracy
dt_test_accuracy = DT.score(x_test,y_test)
print("Testing accuracy = ",DT.score(x_test,y_test))

In [ ]:
y_pred = DT.predict(x_test)
y_true = y_test

from sklearn.metrics import confusion_matrix

cm_dt = confusion_matrix(y_true, y_pred)
labels = list(label_to_number.keys())

f, ax = plt.subplots(figsize=(15,10))
sns.heatmap(cm_dt, annot=True, linewidth=0.5, cmap='viridis', ax=ax,
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title('Predicted vs Actual (Decision Tree Model)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(n_estimators=20, random_state=0)
RF.fit(x_train,y_train)

predicted_values = RF.predict(x_test)

x = metrics.accuracy_score(y_test, predicted_values)
acc.append(x)
model.append('Random Forest')
print("Random Forest Accuracy is: ", x)

print(classification_report(y_test, predicted_values))

In [ ]:
score = cross_val_score(RF,features,target,cv=5)
print('Cross validation score: ',score)

#Print Train Accuracy
rf_train_accuracy = RF.score(x_train,y_train)
print("Training accuracy = ",RF.score(x_train,y_train))
#Print Test Accuracy
rf_test_accuracy = RF.score(x_test,y_test)
print("Testing accuracy = ",RF.score(x_test,y_test))

In [ ]:
y_pred = RF.predict(x_test)
y_true = y_test

from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_true, y_pred)
labels = list(label_to_number.keys())

f, ax = plt.subplots(figsize=(15,10))
sns.heatmap(cm_rf, annot=True, linewidth=0.5, cmap='viridis', ax=ax,
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title('Predicted vs Actual (Random Forest Model)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

## Comparing Models KNN, Decision Tree, Random Forest

In [ ]:
plt.figure(figsize=[14,7],dpi = 100, facecolor='white')
plt.title('Accuracy Comparison')
plt.xlabel('Accuracy')
plt.ylabel('ML Algorithms')
sns.barplot(x=acc, y=model, hue=model, palette='viridis', legend=False)
plt.savefig('plot.png', dpi=300, bbox_inches='tight')

📌 __Penjelasan__:

Membandingkan akurasi setiap model terhadap dataset.

In [ ]:
label = ['KNN', 'Decision Tree','Random Forest']
Test = [knn_test_accuracy, dt_test_accuracy,rf_test_accuracy]
Train = [knn_train_accuracy,  dt_train_accuracy, rf_train_accuracy]

f, ax = plt.subplots(figsize=(20,7))
X_axis = np.arange(len(label))
plt.bar(X_axis - 0.2,Test, 0.4, label = 'Test', color=('midnightblue'))
plt.bar(X_axis + 0.2,Train, 0.4, label = 'Train', color=('mediumaquamarine'))

plt.xticks(X_axis, label)
plt.xlabel("ML algorithms")
plt.ylabel("Accuracy")
plt.title("Testing vs Training Accuracy")
plt.legend()
plt.show()

📌 __Penjelasan__:

Membandingkan akurasi testing dan akurasi training dari setiap model dengan grafik bar.

### Inference

In [ ]:
columns = ['nitrogen', 'phosphorus', 'kalium', 'temperature', 'humidity', 'ph', 'rainfall']
data_sample=[90, 42, 43, 20.8, 82.3, 6.5, 202.93]

inference_test = pd.DataFrame([data_sample], columns=columns)

print("Input data:")
print(inference_test)

# Get predictions and convert to string labels
knn_pred = label_encoder.inverse_transform(knn_1.predict(inference_test))
dt_pred = label_encoder.inverse_transform(DT.predict(inference_test))
rf_pred = label_encoder.inverse_transform(RF.predict(inference_test))

print(f"\nPredictions:")
print(f"K-Nearest Neighbor: {knn_pred}")
print(f"Decision Tree: {dt_pred}")
print(f"Random Forest: {rf_pred}")

📌 __Penjelasan__:

melakukan simulasi inferensi/prediksi menggunakan data input baru dan melihat hasil prediksi dari masing-masing model machine learning yang sudah dilatih.